# E1-04 · Laboratorio 3 · Modelos de probabilidad aplicados a datos reales

**Estadística I · Ingeniería Industrial · Universidad Industrial de Santander**  
*Versión docente (con soluciones)*

**Propósito de aprendizaje**

- Aplicar los modelos normal, uniforme, binomial y exponencial a variables de una base de datos real.
- Explorar y depurar variables numéricas antes de modelarlas.
- Comparar probabilidades estimadas por un modelo con las observadas en los datos y argumentar cuándo un modelo es útil.

**Duración sugerida:** 2 sesiones de 2 horas  
**Datos:** Embarazo en adolescentes, Datos Abiertos Colombia (`dpcy-dq5j`)

## Caso de estudio

La Secretaría de Salud de un departamento colombiano desarrolla una estrategia de seguimiento y prevención del embarazo en adolescentes. Para ello necesita analizar información histórica sobre nacimientos en madres adolescentes, comprender algunas características de esta población y apoyar decisiones en salud pública, acompañamiento familiar y educación sexual.

La entidad dispone de la base **“Embarazo en adolescentes”**, publicada en el portal de [Datos Abiertos Colombia](https://www.datos.gov.co/Salud-y-Protecci-n-Social/Embarazo-en-adolescentes/dpcy-dq5j). Contiene, entre otras, la edad de la madre, la edad del padre, el peso y la talla del bebé al nacer. La Secretaría pide un análisis probabilístico que responda:

1. ¿Cómo se distribuyen la edad de la madre, la edad del padre, el peso y la talla del bebé?
2. ¿Qué variables se aproximan mejor a una distribución normal?
3. ¿El modelo uniforme representa adecuadamente la edad de la madre?
4. ¿Cuál es la probabilidad de encontrar bebés con bajo peso al nacer en una muestra pequeña?
5. ¿Cuál es la probabilidad de encontrar madres de 15 años o menos en una muestra pequeña?
6. ¿La diferencia entre la edad del padre y la de la madre puede analizarse con un modelo exponencial?

El análisis **no** pretende afirmar que los datos siguen perfectamente una distribución teórica. Los modelos son herramientas para interpretar los datos, estimar probabilidades y comprender mejor el fenómeno.

**Instrucciones:** ejecute las celdas en orden. Después de cada análisis escriba su interpretación en la caja de texto y pulse **Guardar respuesta**. Al final descargue el archivo Word con sus respuestas y entréguelo según indique su docente.

> **📌 Nota para el docente**
>
> **Antes de la clase:** ejecute el notebook completo con la versión vigente de la base (las cifras cambian cuando la entidad actualiza el portal). Las respuestas modelo de esta versión se calculan con los datos que usted cargue, de modo que siempre coinciden con lo que verán los estudiantes.
>
> **Puntos de discusión que suelen aparecer:**
> - La base trae **códigos especiales** (por ejemplo 99 o 999 para "sin información") que inflan máximos y medias. Es la razón de la sección de depuración.
> - Peso y talla se registran con pocos decimales (la talla casi siempre en centímetros enteros). Eso produce histogramas "con dientes" y explica pequeñas diferencias entre la probabilidad normal y la observada; es un buen momento para hablar de la corrección por continuidad.
> - El modelo **uniforme** se incluye como contraste: se espera que **no** ajuste, porque los nacimientos se concentran en las edades mayores del rango adolescente.
> - El modelo **exponencial** para la diferencia de edad es discutible: la moda de la diferencia no está en cero (los padres suelen ser algunos años mayores). Es una buena oportunidad para que el estudiante critique un modelo, que es lo que evalúa el criterio 4 de la rúbrica.
> - La rúbrica (criterio 2) menciona los modelos normal, binomial y exponencial; el uniforme se trabaja como contraste.

# 1. Preparación del entorno

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import unicodedata
from scipy.stats import norm, uniform, binom, expon

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.titlesize": 13, "axes.labelsize": 11,
                     "axes.grid": True, "grid.alpha": .3})
AZUL, CORAL = "#00ADC6", "#FF6F49"

In [ ]:
# Funciones de apoyo para la presentación y para registrar respuestas
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

respuestas = {}

def bloque(titulo, texto, color="#00838f", fondo="#e6f7f9"):
    display(HTML(f"""
    <div style="background:{fondo};color:#1f2937;padding:14px 18px;border-radius:12px;border-left:6px solid {color};
                font-family:Arial, sans-serif;margin:10px 0;">
      <h3 style="color:{color};margin:0 0 6px 0;">{titulo}</h3>
      <p style="font-size:15px;line-height:1.55;margin:0;text-align:justify;">{texto}</p>
    </div>"""))

def pregunta(id_pregunta, enunciado):
    """Muestra una pregunta de análisis con una caja de texto y un botón para guardar la respuesta."""
    caja = widgets.Textarea(value=respuestas.get(id_pregunta, ""), placeholder="Escriba aquí su análisis...",
                            layout=widgets.Layout(width="100%", height="130px"))
    boton = widgets.Button(description="Guardar respuesta", button_style="success", icon="check")
    salida = widgets.Output()
    def guardar(_):
        respuestas[id_pregunta] = caja.value
        with salida:
            clear_output(); print("Respuesta guardada.")
    boton.on_click(guardar)
    display(HTML(f"""
    <div style="background:#fff4f0;color:#1f2937;padding:12px 16px;border-radius:10px;border:1px solid #ffc9b8;
                font-family:Arial, sans-serif;margin:10px 0;">
      <b style="color:#d9480f;">Pregunta de análisis {id_pregunta}</b>
      <p style="font-size:15px;line-height:1.5;margin:6px 0 0 0;">{enunciado}</p>
    </div>"""))
    display(caja, boton, salida)

bloque("Instrucción inicial",
       "Ejecute las celdas en orden. Después de cada análisis escriba su interpretación y pulse <b>Guardar respuesta</b>.")

# 2. Carga de la base de datos

Elija en el formulario cómo cargar la base. La opción `descargar` la trae directamente del portal; si la conexión falla, descárguela manualmente en formato **CSV** desde el [portal](https://www.datos.gov.co/Salud-y-Protecci-n-Social/Embarazo-en-adolescentes/dpcy-dq5j) (botón *Exportar → CSV*) y use la opción `subir`.

In [ ]:
#@markdown ### **¿Cómo quiere cargar la base?**
#@markdown - `descargar`: la descarga directamente desde datos.gov.co (necesita internet; puede tardar).
#@markdown - `subir`: abre una ventana para subir el CSV que ya descargó (solo en Colab).
#@markdown - `archivo`: lee un CSV que ya está en la carpeta del notebook.
metodo = "descargar"  #@param ["descargar", "subir", "archivo"]
nombre_archivo = ""  #@param {type:"string"}
#@markdown ---
URL = "https://www.datos.gov.co/api/views/dpcy-dq5j/rows.csv?accessType=DOWNLOAD"

if metodo == "descargar":
    print("Descargando Embarazo en adolescentes desde datos.gov.co ...")
    df = pd.read_csv(URL, low_memory=False)
elif metodo == "subir":
    from google.colab import files
    subido = files.upload()
    df = pd.read_csv(next(iter(subido)), low_memory=False)
else:
    df = pd.read_csv(nombre_archivo, sep=None, engine="python")

print("Registros:", len(df), "| Variables:", df.shape[1])
df.head()

# 3. Limpieza de nombres de columnas y selección de variables

In [ ]:
def limpiar_nombre_columna(col):
    col = unicodedata.normalize("NFKC", str(col)).replace("\ufeff", "").replace("\xa0", " ")
    return col.strip().upper().replace(" ", "_")

df.columns = [limpiar_nombre_columna(c) for c in df.columns]
print("Columnas disponibles:")
print(list(df.columns))

In [ ]:
col_edad_madre, col_edad_padre = "EDAD_MADRE", "EDAD_PADRE"
col_peso, col_talla = "PESO_NAC", "TALLA_NAC"
columnas_interes = [col_edad_madre, col_edad_padre, col_peso, col_talla]

faltan = [c for c in columnas_interes if c not in df.columns]
if faltan:
    raise KeyError(f"No se encontraron {faltan}. Revise la lista de columnas de arriba y corrija los nombres en esta celda.")

df_modelos = df[columnas_interes].apply(pd.to_numeric, errors="coerce")
print("Registros:", len(df_modelos))
pd.DataFrame({"disponibles": df_modelos.notna().sum(), "faltantes": df_modelos.isna().sum()})

# 4. Exploración inicial sin depuración

Antes de depurar, observe las variables **tal como vienen en la base original**. Así se identifican valores sin sentido para el análisis, como códigos especiales o registros atípicos que distorsionan los resultados.

In [ ]:
bloque("Antes de depurar: observemos los datos originales",
       "En bases reales aparecen valores especiales o atípicos. Antes de corregirlos, revise el resumen descriptivo y los valores más altos de cada variable.",
       color="#b45309", fondo="#fff7ed")
display(df_modelos.describe().round(2))
for col in columnas_interes:
    print(f"{col}: 10 valores más altos y su frecuencia")
    print(df_modelos[col].value_counts().sort_index().tail(10).to_string(), "\n")

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(12, 7))
for ax, col in zip(ejes.ravel(), columnas_interes):
    ax.hist(df_modelos[col].dropna(), bins=30, color="#999999")
    ax.set_title(f"{col} (sin depurar)")
plt.tight_layout(); plt.show()

> **✅ Respuesta modelo (docente)**
>
> Los máximos de las edades (y a veces de peso y talla) son valores imposibles, como 99 o 999, que corresponden a **códigos de "sin información"** y no a mediciones. También pueden aparecer edades del padre por debajo de 10 años o pesos de 0. Si no se eliminan, la media y la desviación estándar quedan infladas y los modelos (que usan esos parámetros) asignan probabilidades erróneas. Además, el histograma muestra barras aisladas lejos del resto de los datos.

In [ ]:
for col in columnas_interes:
    s = df_modelos[col].dropna()
    print(f"{col}: mín = {s.min()}, máx = {s.max()}, valores más frecuentes en la cola: {s.value_counts().sort_index().tail(3).to_dict()}")

## Depuración por variable

En lugar de eliminar filas completas, se depura **cada variable por separado** con rangos plausibles. Así se conserva la mayor cantidad de información: si falta la edad del padre, la fila sigue sirviendo para analizar el peso del bebé.

| Variable | Rango válido | Justificación |
|---|---|---|
| Edad de la madre | 10 a 19 años | definición de embarazo adolescente |
| Edad del padre | 10 a 80 años | excluye códigos y errores de digitación |
| Peso al nacer | 0,5 a 6 kg | rango biológicamente plausible |
| Talla al nacer | 20 a 70 cm | rango biológicamente plausible |

In [ ]:
rangos = {col_edad_madre: (10, 19), col_edad_padre: (10, 80), col_peso: (0.5, 6), col_talla: (20, 70)}
df_modelos_limpio = df_modelos.copy()
for col, (li, ls) in rangos.items():
    df_modelos_limpio[col] = df_modelos_limpio[col].where(df_modelos_limpio[col].between(li, ls))

pd.DataFrame({"válidos antes": df_modelos.notna().sum(), "válidos después": df_modelos_limpio.notna().sum(),
              "descartados": df_modelos.notna().sum() - df_modelos_limpio.notna().sum()})

In [ ]:
display(df_modelos_limpio.describe().round(3))

# 5. Exploración de las variables depuradas

In [ ]:
nombres = {col_edad_madre: "Edad de la madre (años)", col_edad_padre: "Edad del padre (años)",
           col_peso: "Peso del bebé al nacer (kg)", col_talla: "Talla del bebé al nacer (cm)"}
fig, ejes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (col, nombre) in zip(ejes.ravel(), nombres.items()):
    ax.hist(df_modelos_limpio[col].dropna(), bins=20, color=AZUL)
    ax.set_title(nombre)
plt.tight_layout(); plt.show()
print("Asimetría:", df_modelos_limpio.skew().round(3).to_dict())

> **✅ Respuesta modelo (docente)**
>
> El **peso** y la **talla** del bebé tienen forma de campana, con una cola algo más larga hacia la izquierda (bebés prematuros o de bajo peso); son las candidatas naturales al modelo normal. La **edad de la madre** está concentrada en los valores altos del rango (16 a 19 años) y es asimétrica a la izquierda: no es simétrica ni uniforme. La **edad del padre** es asimétrica a la derecha: la mayoría está entre los 18 y los 25 años, pero hay padres mucho mayores.

In [ ]:
a = df_modelos_limpio.skew()
print("Asimetrías calculadas con sus datos:", a.round(2).to_dict())
print("Variable más simétrica:", a.abs().idxmin(), "| más asimétrica:", a.abs().idxmax())

# 6. Análisis 1 · Modelo normal para el peso al nacer

**Pregunta:** ¿el peso del bebé al nacer se aproxima a una distribución normal? Se estiman μ y σ con la media y la desviación estándar muestrales y se comparan dos probabilidades: el **bajo peso** (≤ 2,5 kg, definición de la OMS) y el rango de 3 a 4 kg.

In [ ]:
datos_peso = df_modelos_limpio[col_peso].dropna()
media_peso, desv_peso = datos_peso.mean(), datos_peso.std()
print(f"n = {len(datos_peso)}  |  media = {media_peso:.3f} kg  |  desviación = {desv_peso:.3f} kg")

x = np.linspace(datos_peso.min(), datos_peso.max(), 300)
plt.hist(datos_peso, bins=25, density=True, color=AZUL, alpha=.55, label="datos observados")
plt.plot(x, norm.pdf(x, media_peso, desv_peso), color=CORAL, lw=2.5, label=f"Normal({media_peso:.2f}; {desv_peso:.2f})")
plt.title("Peso al nacer vs. modelo normal"); plt.xlabel("kg"); plt.ylabel("densidad"); plt.legend(); plt.show()

In [ ]:
P_bajo_peso_normal = norm.cdf(2.5, media_peso, desv_peso)
P_peso_3_4_normal = norm.cdf(4, media_peso, desv_peso) - norm.cdf(3, media_peso, desv_peso)
P_bajo_peso_observada = (datos_peso <= 2.5).mean()
P_peso_3_4_observada = datos_peso.between(3, 4).mean()

pd.DataFrame({"modelo normal": [P_bajo_peso_normal, P_peso_3_4_normal],
              "observada en los datos": [P_bajo_peso_observada, P_peso_3_4_observada]},
             index=["P(peso ≤ 2,5 kg)", "P(3 ≤ peso ≤ 4 kg)"]).style.format("{:.2%}")

> **✅ Respuesta modelo (docente)**
>
> Un buen modelo produce probabilidades **cercanas** a las observadas. Para el peso, el modelo normal suele reproducir bien la zona central (3 a 4 kg) con diferencias de pocos puntos porcentuales. En la cola izquierda (bajo peso) la diferencia relativa es mayor porque la distribución real tiene una cola izquierda más pesada que la normal (partos prematuros). Conclusión esperada: el modelo normal es **adecuado para describir el comportamiento típico** del peso, pero hay que usarlo con cautela para estimar la frecuencia de bajo peso, que es justamente un indicador de salud pública.

In [ ]:
d1, d2 = P_bajo_peso_normal - P_bajo_peso_observada, P_peso_3_4_normal - P_peso_3_4_observada
print(f"Con sus datos: bajo peso → modelo {P_bajo_peso_normal:.1%} vs. observado {P_bajo_peso_observada:.1%} (diferencia {d1:+.1%}).")
print(f"               3 a 4 kg  → modelo {P_peso_3_4_normal:.1%} vs. observado {P_peso_3_4_observada:.1%} (diferencia {d2:+.1%}).")
print("Lectura:", "el modelo subestima" if d1 < 0 else "el modelo sobreestima", "la proporción de bajo peso.")

# 7. Análisis 2 · Modelo normal para la talla al nacer

**Pregunta:** ¿la talla del bebé al nacer se aproxima a una distribución normal?

In [ ]:
datos_talla = df_modelos_limpio[col_talla].dropna()
media_talla, desv_talla = datos_talla.mean(), datos_talla.std()
print(f"n = {len(datos_talla)}  |  media = {media_talla:.2f} cm  |  desviación = {desv_talla:.2f} cm")

x = np.linspace(datos_talla.min(), datos_talla.max(), 300)
plt.hist(datos_talla, bins=25, density=True, color=AZUL, alpha=.55, label="datos observados")
plt.plot(x, norm.pdf(x, media_talla, desv_talla), color=CORAL, lw=2.5, label=f"Normal({media_talla:.1f}; {desv_talla:.1f})")
plt.title("Talla al nacer vs. modelo normal"); plt.xlabel("cm"); plt.ylabel("densidad"); plt.legend(); plt.show()

In [ ]:
P_talla_45_normal = norm.cdf(45, media_talla, desv_talla)
P_talla_48_52_normal = norm.cdf(52, media_talla, desv_talla) - norm.cdf(48, media_talla, desv_talla)
P_talla_45_observada = (datos_talla <= 45).mean()
P_talla_48_52_observada = datos_talla.between(48, 52).mean()

pd.DataFrame({"modelo normal": [P_talla_45_normal, P_talla_48_52_normal],
              "observada en los datos": [P_talla_45_observada, P_talla_48_52_observada]},
             index=["P(talla ≤ 45 cm)", "P(48 ≤ talla ≤ 52 cm)"]).style.format("{:.2%}")

> **✅ Respuesta modelo (docente)**
>
> La talla se registra casi siempre en centímetros enteros, de modo que muchos bebés comparten exactamente el mismo valor (48, 49, 50...). Por eso la probabilidad observada del intervalo 48–52 cm suele ser **mayor** que la del modelo: el intervalo cerrado incluye por completo los valores 48 y 52, mientras que el modelo continuo reparte esa masa. La diferencia no indica que la talla "no sea normal", sino que la variable está **discretizada**; con la corrección por continuidad (calcular entre 47,5 y 52,5) las dos probabilidades se acercan. La forma general sí es acampanada, por lo que el modelo normal es útil como aproximación.

In [ ]:
cc = norm.cdf(52.5, media_talla, desv_talla) - norm.cdf(47.5, media_talla, desv_talla)
print(f"Sin corrección: modelo {P_talla_48_52_normal:.1%} vs. observado {P_talla_48_52_observada:.1%}")
print(f"Con corrección por continuidad (47,5 a 52,5): modelo {cc:.1%}")
print(f"Proporción de tallas enteras en la base: {(datos_talla == datos_talla.round()).mean():.1%}")

# 8. Análisis 3 · Modelo uniforme como contraste para la edad de la madre

**Pregunta:** ¿la edad de la madre se comporta como si todos los valores dentro del rango fueran igualmente probables?

Este análisis es un **contraste**: el objetivo no es forzar el ajuste, sino verificar si el modelo representa o no lo observado.

In [ ]:
datos_madre = df_modelos_limpio[col_edad_madre].dropna()
a_m, b_m = datos_madre.min(), datos_madre.max()

frec = datos_madre.round().value_counts(normalize=True).sort_index()
plt.bar(frec.index, frec.values, color=AZUL, alpha=.7, label="proporción observada por edad")
plt.axhline(1 / (b_m - a_m + 1), color=CORAL, lw=2.5, label="uniforme discreta (todas las edades iguales)")
plt.title("Edad de la madre vs. modelo uniforme"); plt.xlabel("años"); plt.ylabel("proporción"); plt.legend(); plt.show()

P_madre_15_uniforme = uniform.cdf(15, loc=a_m, scale=b_m - a_m)
P_madre_15_observada = (datos_madre <= 15).mean()
print(f"P(edad ≤ 15) según el modelo uniforme continuo en [{a_m:.0f}, {b_m:.0f}]: {P_madre_15_uniforme:.2%}")
print(f"P(edad ≤ 15) observada:                                   {P_madre_15_observada:.2%}")

> **✅ Respuesta modelo (docente)**
>
> **No.** Si la edad fuera uniforme, todas las barras tendrían la altura de la línea horizontal. En cambio, la proporción crece con la edad: hay muy pocas madres de 10 a 13 años y la mayoría tiene entre 16 y 19. El modelo uniforme **sobreestima** mucho la probabilidad de madres de 15 años o menos. La lección es que un modelo debe elegirse por la forma de los datos y no solo porque la variable esté acotada en un intervalo.

In [ ]:
print(f"Con sus datos el modelo uniforme da {P_madre_15_uniforme:.1%} y lo observado es {P_madre_15_observada:.1%}: el modelo multiplica por {P_madre_15_uniforme / P_madre_15_observada:.1f} la proporción real.")

# 9. Análisis 4 · Modelo binomial para el bajo peso al nacer

**Pregunta:** si se seleccionan al azar 10 registros, ¿cuál es la probabilidad de encontrar cierto número de bebés con bajo peso?

- Ensayo: un nacimiento elegido al azar. **Éxito:** peso ≤ 2,5 kg.
- p = proporción de bajo peso en la base; n = 10. Entonces X = número de bebés con bajo peso ~ Binomial(10, p).

Para comparar con los datos, además se **simulan** 5.000 muestras de 10 registros tomados de la base y se cuenta cuántos bebés con bajo peso aparecen en cada una.

In [ ]:
n_muestra = 10
p_bajo_peso = (datos_peso <= 2.5).mean()
k = np.arange(n_muestra + 1)
binomial_peso = binom.pmf(k, n_muestra, p_bajo_peso)

rng = np.random.default_rng(2026)
simulados = np.array([(rng.choice(datos_peso.to_numpy(), n_muestra) <= 2.5).sum() for _ in range(5000)])
observado_peso = np.bincount(simulados, minlength=n_muestra + 1) / len(simulados)

plt.bar(k - .2, binomial_peso, width=.4, color=CORAL, label=f"Binomial(10; {p_bajo_peso:.3f})")
plt.bar(k + .2, observado_peso, width=.4, color=AZUL, label="muestras de la base (simulación)")
plt.xticks(k); plt.xlabel("número de bebés con bajo peso en 10 registros"); plt.ylabel("probabilidad")
plt.title("Bebés con bajo peso en muestras de 10"); plt.legend(); plt.show()

tabla_peso = pd.DataFrame({"binomial": binomial_peso, "simulación": observado_peso}, index=k).head(6)
print(f"p (bajo peso) = {p_bajo_peso:.4f}")
print(f"P(X = 0) = {binom.pmf(0, 10, p_bajo_peso):.4f}   P(X = 2) = {binom.pmf(2, 10, p_bajo_peso):.4f}   P(X ≥ 2) = {binom.sf(1, 10, p_bajo_peso):.4f}")
tabla_peso.style.format("{:.4f}")

> **✅ Respuesta modelo (docente)**
>
> La moda de la binomial está en el entero cercano a 10·p. Con una proporción de bajo peso cercana al 10 %, lo más probable es encontrar **0 o 1** bebé con bajo peso en 10 registros, y la probabilidad de encontrar 2 o más es pequeña pero no despreciable. Que la simulación con los datos reales coincida con la binomial confirma que los supuestos (ensayos independientes, misma probabilidad de éxito) son razonables al muestrear al azar. Interpretación: el bajo peso no es la situación típica, pero en un grupo pequeño de nacimientos es frecuente que aparezca al menos un caso, lo que justifica el seguimiento.

In [ ]:
print(f"Moda binomial con sus datos: {k[binomial_peso.argmax()]} bebés (P = {binomial_peso.max():.3f}); P(al menos uno) = {1 - binomial_peso[0]:.3f}")

# 10. Análisis 5 · Modelo binomial para madres de 15 años o menos

**Pregunta:** si se seleccionan 10 registros, ¿cuál es la probabilidad de encontrar cierto número de madres de 15 años o menos? **Éxito:** madre con 15 años o menos.

In [ ]:
p_madre_15 = (datos_madre <= 15).mean()
binomial_madre = binom.pmf(k, n_muestra, p_madre_15)
simulados = np.array([(rng.choice(datos_madre.to_numpy(), n_muestra) <= 15).sum() for _ in range(5000)])
observado_madre = np.bincount(simulados, minlength=n_muestra + 1) / len(simulados)

plt.bar(k - .2, binomial_madre, width=.4, color=CORAL, label=f"Binomial(10; {p_madre_15:.3f})")
plt.bar(k + .2, observado_madre, width=.4, color=AZUL, label="muestras de la base (simulación)")
plt.xticks(k); plt.xlabel("número de madres de 15 años o menos en 10 registros"); plt.ylabel("probabilidad")
plt.title("Madres de 15 años o menos en muestras de 10"); plt.legend(); plt.show()
print(f"p = {p_madre_15:.4f}   P(X = 3) = {binom.pmf(3, 10, p_madre_15):.4f}   P(X ≥ 1) = {binom.sf(0, 10, p_madre_15):.4f}")

> **✅ Respuesta modelo (docente)**
>
> La moda suele estar en **0 o 1** madre de 15 años o menos, porque esa proporción es baja. Esto es coherente con el histograma de la edad de la madre: las barras de 10 a 15 años son bajas y la mayoría de nacimientos ocurre entre los 16 y los 19 años. El resultado también muestra por qué el modelo uniforme falla: si la edad fuera uniforme, p sería mucho mayor y la binomial se desplazaría hacia la derecha.

In [ ]:
print(f"Moda con sus datos: {k[binomial_madre.argmax()]} madres; con p uniforme ({P_madre_15_uniforme:.2f}) la moda sería {k[binom.pmf(k, 10, P_madre_15_uniforme).argmax()]}.")

# 11. Análisis 6 · Modelo exponencial para la diferencia de edad entre padre y madre

**Pregunta:** ¿la diferencia absoluta entre la edad del padre y la de la madre puede modelarse con una distribución exponencial? El modelo exponencial describe variables positivas en las que los valores pequeños son los más frecuentes. Su parámetro se estima con la media: λ = 1 / x̄.

In [ ]:
par = df_modelos_limpio[[col_edad_madre, col_edad_padre]].dropna()
diferencia_padres = (par[col_edad_padre] - par[col_edad_madre]).abs()
media_dif = diferencia_padres.mean()
print(f"n = {len(diferencia_padres)}  |  media = {media_dif:.2f} años  |  λ = {1/media_dif:.4f}")
display(diferencia_padres.describe().round(2).to_frame("diferencia de edad").T)

x = np.linspace(0, diferencia_padres.quantile(.995), 300)
plt.hist(diferencia_padres, bins=range(0, int(diferencia_padres.quantile(.995)) + 2), density=True, color=AZUL, alpha=.55, label="datos observados")
plt.plot(x, expon.pdf(x, scale=media_dif), color=CORAL, lw=2.5, label=f"Exponencial(λ = {1/media_dif:.3f})")
plt.title("Diferencia de edad entre padre y madre vs. modelo exponencial"); plt.xlabel("años"); plt.ylabel("densidad"); plt.legend(); plt.show()

In [ ]:
P_dif_5_modelo = expon.cdf(5, scale=media_dif)
P_dif_10_modelo = expon.sf(10, scale=media_dif)
P_dif_5_observada = (diferencia_padres <= 5).mean()
P_dif_10_observada = (diferencia_padres > 10).mean()
pd.DataFrame({"modelo exponencial": [P_dif_5_modelo, P_dif_10_modelo],
              "observada en los datos": [P_dif_5_observada, P_dif_10_observada]},
             index=["P(diferencia ≤ 5 años)", "P(diferencia > 10 años)"]).style.format("{:.2%}")

> **✅ Respuesta modelo (docente)**
>
> El modelo exponencial supone que la diferencia **más frecuente es cero** y que la frecuencia decrece desde allí. En los datos, en cambio, la moda está algunos años por encima de cero (el padre suele ser entre 2 y 6 años mayor) y hay pocas parejas con exactamente la misma edad. Por eso el modelo **sobreestima** la frecuencia de diferencias muy pequeñas, aunque puede aproximar razonablemente la cola derecha (diferencias grandes). Conclusión: la exponencial capta la asimetría a la derecha, pero no la forma cerca de cero; un modelo gamma o lognormal sería más adecuado (puede probarse con el notebook H2).

In [ ]:
moda = diferencia_padres.round().mode().iat[0]
print(f"Moda observada de la diferencia: {moda:.0f} años; P(dif ≤ 1) modelo = {expon.cdf(1, scale=media_dif):.1%} vs. observado = {(diferencia_padres <= 1).mean():.1%}")

# 12. Tabla resumen

In [ ]:
resumen_modelos = pd.DataFrame({
    "modelo": ["Normal", "Normal", "Uniforme", "Binomial (n = 10)", "Binomial (n = 10)", "Exponencial"],
    "variable": ["Peso al nacer", "Talla al nacer", "Edad de la madre", "Peso al nacer", "Edad de la madre", "Diferencia de edad padre–madre"],
    "evento": ["peso ≤ 2,5 kg", "talla ≤ 45 cm", "edad ≤ 15 años", "al menos 1 bebé con bajo peso", "al menos 1 madre ≤ 15 años", "diferencia ≤ 5 años"],
    "probabilidad del modelo": [P_bajo_peso_normal, P_talla_45_normal, P_madre_15_uniforme,
                                binom.sf(0, 10, p_bajo_peso), binom.sf(0, 10, p_madre_15), P_dif_5_modelo],
    "probabilidad observada": [P_bajo_peso_observada, P_talla_45_observada, P_madre_15_observada,
                               1 - observado_peso[0], 1 - observado_madre[0], P_dif_5_observada],
})
resumen_modelos["diferencia"] = resumen_modelos["probabilidad del modelo"] - resumen_modelos["probabilidad observada"]
resumen_modelos.style.format({c: "{:.2%}" for c in ["probabilidad del modelo", "probabilidad observada", "diferencia"]})

> En las filas binomiales la probabilidad observada proviene de la simulación con muestras de 10 registros de la base.

> **✅ Respuesta modelo (docente)**
>
> Una buena conclusión (a) identifica que el **modelo normal** es el más útil para peso y talla, con la advertencia sobre las colas y la discretización; (b) reconoce que el **binomial** funciona bien cuando sus supuestos se cumplen (muestreo aleatorio, p constante) y que la simulación lo confirma; (c) descarta el **uniforme** para la edad de la madre con argumentos basados en el gráfico; (d) critica el **exponencial** porque la moda de la diferencia no está en cero. Entre las limitaciones deben aparecer: códigos especiales y datos faltantes, valores redondeados, parámetros estimados con los mismos datos y el hecho de que un modelo es una simplificación útil y no una descripción exacta.